# Fine-Tune Whisper-Small for Twi/Akan ASR

Fine-tunes OpenAI Whisper-Small on UGSpeechData (Akan) + Mozilla Common Voice Twi.
This gives you a self-hosted Twi ASR model that replaces Khaya STT calls.

**Recommended runtime:** TPU v2/v3 (Colab Pro+) or A100 GPU
**Estimated training time:** 4–8 hours on TPU v3

**Expected outcome:** WER ~20–30% on spontaneous Twi speech
(vs ~37% for out-of-box Whisper on Akan Bible data)

**After training:** Push model to Hugging Face Hub, set WHISPER_AKAN_URL in .env,
update registry.ts status to 'active'.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
HF_TOKEN = ""             # Hugging Face token (for pushing model)
HF_REPO = "your-username/whisper-small-twi"  # Where to push the trained model

MODEL_NAME = "openai/whisper-small"  # Base model
LANGUAGE = "Twi"
TASK = "transcribe"

# Dataset mix: UGSpeechData (primary) + Common Voice Twi (supplementary)
# UGSpeechData: 104h transcribed Akan — https://huggingface.co/datasets/UGSpeech/UGSpeechData
# Common Voice: community Twi audio — https://huggingface.co/datasets/mozilla-foundation/common_voice_13_0
USE_UGSPEECHDATA = True
USE_COMMON_VOICE = True

# Training hyperparameters (validated for Whisper-Small fine-tuning)
LEARNING_RATE = 1.25e-5   # Pre-train LR (5e-4) / 40 = fine-tune LR
TRAIN_STEPS = 4000
EVAL_STEPS = 500
BATCH_SIZE = 16
WARMUP_STEPS = 400
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
!pip install -q transformers datasets accelerate evaluate jiwer huggingface_hub
!pip install -q git+https://github.com/huggingface/peft.git
print("Packages installed.")

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)
print("Logged in to Hugging Face.")

In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, DatasetDict, Audio

datasets_to_merge = []

if USE_UGSPEECHDATA:
    print("Loading UGSpeechData (Akan)...")
    # NOTE: Check the exact dataset ID on HF Hub — may require access request
    ug_data = load_dataset("UGSpeech/UGSpeechData", "akan", trust_remote_code=True)
    # Standardize column names
    ug_data = ug_data.rename_columns({"audio": "audio", "sentence": "sentence"})
    datasets_to_merge.append(ug_data)
    print(f"UGSpeechData: {ug_data}")

if USE_COMMON_VOICE:
    print("Loading Mozilla Common Voice (Twi)...")
    cv_data = load_dataset(
        "mozilla-foundation/common_voice_13_0",
        "tw",
        split="train+validation",
        use_auth_token=True
    )
    datasets_to_merge.append(cv_data)
    print(f"Common Voice Twi: {len(cv_data)} samples")

In [ ]:
# ── Feature extraction ────────────────────────────────────────────────────
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

def prepare_dataset(batch):
    # Resample to 16kHz (Whisper requirement)
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

print("Feature extractor loaded.")

In [ ]:
# Process datasets (this takes ~20-40 min for the full UGSpeechData)
# Use num_proc to parallelize on TPU

# Use a subset first to validate the pipeline
QUICK_RUN = True  # Set to False for full training
QUICK_SAMPLES = 500

if USE_UGSPEECHDATA:
    train_data = ug_data["train"]
    eval_data = ug_data["validation"] if "validation" in ug_data else ug_data["test"]

    if QUICK_RUN:
        train_data = train_data.select(range(min(QUICK_SAMPLES, len(train_data))))
        eval_data = eval_data.select(range(min(100, len(eval_data))))

    train_data = train_data.cast_column("audio", Audio(sampling_rate=16000))
    eval_data = eval_data.cast_column("audio", Audio(sampling_rate=16000))

    train_data = train_data.map(prepare_dataset, remove_columns=train_data.column_names, num_proc=4)
    eval_data = eval_data.map(prepare_dataset, remove_columns=eval_data.column_names, num_proc=4)

print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

In [ ]:
# ── Data collator ─────────────────────────────────────────────────────────
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

In [ ]:
# ── Load model + WER metric ───────────────────────────────────────────────
from transformers import WhisperForConditionalGeneration
import evaluate

model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
model.generation_config.language = LANGUAGE.lower()
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("Model loaded.")

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-twi",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=1,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=TRAIN_STEPS if not QUICK_RUN else 50,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=EVAL_STEPS,
    eval_steps=EVAL_STEPS,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=bool(HF_TOKEN and HF_REPO),
    hub_model_id=HF_REPO if HF_TOKEN else None,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print(f"Starting training ({'QUICK RUN' if QUICK_RUN else 'FULL'}: {training_args.max_steps} steps)...")
trainer.train()

In [ ]:
# ── Evaluate final WER ────────────────────────────────────────────────────
eval_results = trainer.evaluate()
print(f"\nFinal WER: {eval_results['eval_wer']:.1f}%")

# Benchmark interpretation
wer = eval_results['eval_wer']
if wer < 20:
    print("✅ EXCELLENT — ready for production deployment")
elif wer < 35:
    print("✅ GOOD — usable with error-tolerant NLU design")
elif wer < 50:
    print("⚠️  FAIR — needs more domain data or training steps")
else:
    print("❌ POOR — check dataset quality and preprocessing")

In [ ]:
# ── Push to Hugging Face Hub ──────────────────────────────────────────────
if HF_TOKEN and HF_REPO:
    trainer.push_to_hub()
    processor.push_to_hub(HF_REPO)
    print(f"\n✅ Model pushed to https://huggingface.co/{HF_REPO}")
    print("\nNext steps:")
    print(f"1. Set WHISPER_AKAN_URL=https://api-inference.huggingface.co/models/{HF_REPO} in .env")
    print("2. Update lib/registry.ts: set whisper-small-akan status to 'active'")
    print("3. The speech bridge router will prefer it over Khaya for Twi STT")
else:
    print("Model not pushed (no HF_TOKEN set). Save locally:")
    trainer.save_model("./whisper-small-twi-final")